In [18]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, TimestampType
from pyspark.sql.functions import window, sum as spark_sum, count, avg, col
import os
import shutil
import time
import pandas as pd
from datetime import datetime, timedelta

spark = SparkSession.builder \
    .appName("TugasPertemuan10") \
    .master("local[*]") \
    .getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

spark.conf.set("spark.sql.shuffle.partitions", "4")

print("SparkSession siap. Versi Spark:", spark.version)

SparkSession siap. Versi Spark: 3.5.9


In [19]:
folder_tugas = "tugas10_stream"
shutil.rmtree(folder_tugas, ignore_errors=True)
os.makedirs(folder_tugas, exist_ok=True)

skema_tugas = StructType([
    StructField("pesanan_id", StringType(), True),
    StructField("nama_cabang", StringType(), True),
    StructField("nama_menu", StringType(), True),
    StructField("qty", IntegerType(), True),
    StructField("harga_satuan", IntegerType(), True),
    StructField("waktu_pesan", TimestampType(), True)
])

df_stream = spark.readStream \
    .format("csv") \
    .schema(skema_tugas) \
    .option("header", True) \
    .load(folder_tugas)

print("Folder:", folder_tugas, "siap.")
print("df_stream.isStreaming:", df_stream.isStreaming)

Folder: tugas10_stream siap.
df_stream.isStreaming: True


In [20]:
df_stream_dengan_harga = df_stream.withColumn("total_harga", col("qty") * col("harga_satuan"))

agregasi_cabang = df_stream_dengan_harga.groupBy("nama_cabang").agg(
    spark_sum("total_harga").alias("total_pendapatan"),
    count("pesanan_id").alias("jumlah_pesanan")
)

query_cabang = agregasi_cabang.writeStream \
    .outputMode("complete") \
    .format("memory") \
    .queryName("agregasi_cabang") \
    .start()

print("Streaming query 'agregasi_cabang' dimulai! Status:", query_cabang.status)

Streaming query 'agregasi_cabang' dimulai! Status: {'message': 'Waiting for data to arrive', 'isDataAvailable': False, 'isTriggerActive': False}


In [21]:
df_stream2 = spark.readStream \
    .format("csv") \
    .schema(skema_tugas) \
    .option("header", True) \
    .load(folder_tugas)

df_stream2_dengan_harga = df_stream2.withColumn("total_harga", col("qty") * col("harga_satuan"))

agregasi_waktu = df_stream2_dengan_harga.groupBy(
    window(col("waktu_pesan"), "2 minutes")
).agg(
    spark_sum("total_harga").alias("total_pendapatan")
)

query_waktu = agregasi_waktu.writeStream \
    .outputMode("complete") \
    .format("memory") \
    .queryName("agregasi_waktu") \
    .start()

print("Streaming query 'agregasi_waktu' dimulai! Status:", query_waktu.status)

Streaming query 'agregasi_waktu' dimulai! Status: {'message': 'Waiting for data to arrive', 'isDataAvailable': False, 'isTriggerActive': False}


In [23]:
waktu_sekarang = datetime.now()

# BATCH 1
print("Mengirim Batch 1...")
pd.DataFrame({
    "pesanan_id": ["P01", "P02"],
    "nama_cabang": ["Cabang Magelang", "Cabang Jogja"],
    "nama_menu": ["Kopi Susu", "Americano"],
    "qty": [2, 1],
    "harga_satuan": [20000, 15000],
    "waktu_pesan": [waktu_sekarang.strftime("%Y-%m-%d %H:%M:%S")] * 2
}).to_csv("tugas10_stream/batch_tugas_1.csv", index=False)

time.sleep(7) # Jeda waktu agar Spark memproses micro-batch
print("Hasil Agregasi Cabang setelah Batch 1:")
spark.sql("SELECT * FROM agregasi_cabang ORDER BY total_pendapatan DESC").show()

# BATCH 2
print("Mengirim Batch 2...")
pd.DataFrame({
    "pesanan_id": ["P03", "P04"],
    "nama_cabang": ["Cabang Magelang", "Cabang Semarang"],
    "nama_menu": ["Latte", "Espresso"],
    "qty": [1, 3],
    "harga_satuan": [25000, 10000],
    "waktu_pesan": [(waktu_sekarang + timedelta(minutes=1)).strftime("%Y-%m-%d %H:%M:%S")] * 2
}).to_csv("tugas10_stream/batch_tugas_2.csv", index=False)

time.sleep(7)
print("Hasil Agregasi Cabang setelah Batch 2:")
spark.sql("SELECT * FROM agregasi_cabang ORDER BY total_pendapatan DESC").show()


# BATCH 3
print("Mengirim Batch 3...")
pd.DataFrame({
    "pesanan_id": ["P05", "P06"],
    "nama_cabang": ["Cabang Jogja", "Cabang Semarang"],
    "nama_menu": ["Matcha", "Kopi Susu"],
    "qty": [2, 2],
    "harga_satuan": [25000, 20000],
    "waktu_pesan": [(waktu_sekarang + timedelta(minutes=2)).strftime("%Y-%m-%d %H:%M:%S")] * 2
}).to_csv("tugas10_stream/batch_tugas_3.csv", index=False)

time.sleep(7)
print("Hasil Agregasi Cabang setelah Batch 3:")
spark.sql("SELECT * FROM agregasi_cabang ORDER BY total_pendapatan DESC").show()

Mengirim Batch 1...
Hasil Agregasi Cabang setelah Batch 1:
+---------------+----------------+--------------+
|    nama_cabang|total_pendapatan|jumlah_pesanan|
+---------------+----------------+--------------+
|Cabang Magelang|           40000|             1|
|   Cabang Jogja|           15000|             1|
+---------------+----------------+--------------+

Mengirim Batch 2...
Hasil Agregasi Cabang setelah Batch 2:
+---------------+----------------+--------------+
|    nama_cabang|total_pendapatan|jumlah_pesanan|
+---------------+----------------+--------------+
|Cabang Magelang|           65000|             2|
|Cabang Semarang|           30000|             1|
|   Cabang Jogja|           15000|             1|
+---------------+----------------+--------------+

Mengirim Batch 3...
Hasil Agregasi Cabang setelah Batch 3:
+---------------+----------------+--------------+
|    nama_cabang|total_pendapatan|jumlah_pesanan|
+---------------+----------------+--------------+
|Cabang Semarang|    

In [24]:
query3 = df_stream.writeStream \
    .format("parquet") \
    .option("path", "tugas10_output") \
    .option("checkpointLocation", "tugas10_checkpoint") \
    .outputMode("append") \
    .start()

print("Streaming query ke Parquet dimulai! Status:", query3.status)

waktu_sekarang = datetime.now()
print("Mengirim Batch 4...")
pd.DataFrame({
    "pesanan_id": ["P07", "P08"],
    "nama_cabang": ["Cabang Magelang", "Cabang Jogja"],
    "nama_menu": ["Roti Bakar", "Kopi Hitam"],
    "qty": [1, 2],
    "harga_satuan": [15000, 10000],
    "waktu_pesan": [waktu_sekarang.strftime("%Y-%m-%d %H:%M:%S")] * 2
}).to_csv("tugas10_stream/batch_tugas_4.csv", index=False)

time.sleep(10)

print("Isi direktori tugas10_output:")
!ls tugas10_output

print("\nMembaca kembali data dari Parquet:")
df_parquet = spark.read.parquet("tugas10_output")
df_parquet.show()

Streaming query ke Parquet dimulai! Status: {'message': 'Initializing StreamExecution', 'isDataAvailable': False, 'isTriggerActive': False}
Mengirim Batch 4...
Isi direktori tugas10_output:
part-00000-f3208757-c19e-4812-9bd1-75c691976dc3-c000.snappy.parquet
part-00001-787e95a8-9679-416a-b5b0-45313d258dbe-c000.snappy.parquet
part-00002-121edff9-8992-46cd-860f-36c63b6089fb-c000.snappy.parquet
part-00003-8ec4dfaf-0916-4440-9ea5-4a3fb1467a68-c000.snappy.parquet
_spark_metadata

Membaca kembali data dari Parquet:
+----------+---------------+----------+---+------------+-------------------+
|pesanan_id|    nama_cabang| nama_menu|qty|harga_satuan|        waktu_pesan|
+----------+---------------+----------+---+------------+-------------------+
|       P05|   Cabang Jogja|    Matcha|  2|       25000|2026-11-09 10:12:13|
|       P06|Cabang Semarang| Kopi Susu|  2|       20000|2026-11-09 10:12:13|
|       P07|Cabang Magelang|Roti Bakar|  1|       15000|2026-11-09 10:17:45|
|       P08|   Cabang Jo

In [25]:
query_cabang.stop()
query_waktu.stop()
query3.stop()
spark.stop()
print("Streaming query dihentikan, SparkSession ditutup.")

Streaming query dihentikan, SparkSession ditutup.
